In [5]:
# merge_and_split.py
import json
import re
import random
from pathlib import Path

BASE = Path("..")
random.seed(42)
BLOCK_SIZE = 20      # number of consecutive frames per block
TEST_RATIO = 0.2     # test ratio, applied at the block level

sources = [
    {"json": "instances_default.json", "subdir": "kidney_stone", "tag": "video_A"},
    {"json": "new1.json",               "subdir": "new1-jpg",    "tag": "video_B_seg1"},
    {"json": "new2.json",               "subdir": "new2-jpg",    "tag": "video_B_seg2"},
]

def extract_frame_num(filename):
    m = re.search(r"(\d+)", filename)
    return int(m.group(1)) if m else -1

merged_images = []
merged_annotations = []
merged_categories = None
next_image_id = 1
next_ann_id = 1
block_assignments = []  # (source_tag, block_id, split)

for src in sources:
    path = BASE / "training_data" / src["json"]
    with open(path) as f:
        data = json.load(f)

    if merged_categories is None:
        merged_categories = data["categories"]

    # map from old image_id -> new image_id
    old_to_new_id = {}

    # sort by frame number to preserve temporal order for blocking
    images_sorted = sorted(data["images"], key=lambda im: extract_frame_num(im["file_name"]))

    # assign consecutive frames to blocks, then split blocks into train/test
    n_blocks = (len(images_sorted) + BLOCK_SIZE - 1) // BLOCK_SIZE
    block_ids = list(range(n_blocks))
    random.shuffle(block_ids)
    n_test_blocks = max(1, int(n_blocks * TEST_RATIO))
    test_block_set = set(block_ids[:n_test_blocks])

    for idx, img in enumerate(images_sorted):
        block_id = idx // BLOCK_SIZE
        split = "test" if block_id in test_block_set else "train"

        old_id = img["id"]
        new_id = next_image_id
        old_to_new_id[old_id] = new_id
        next_image_id += 1

        new_img = dict(img)
        new_img["id"] = new_id
        new_img["file_name"] = f"{src['subdir']}/{img['file_name']}"  # include subfolder to avoid name collisions
        new_img["source_video"] = src["tag"]
        new_img["split"] = split
        merged_images.append(new_img)

        block_assignments.append({
            "image_id": new_id,
            "file_name": new_img["file_name"],
            "source_video": src["tag"],
            "block_id": f"{src['tag']}_block{block_id}",
            "split": split,
        })

    for ann in data.get("annotations", []):
        new_ann = dict(ann)
        new_ann["id"] = next_ann_id
        new_ann["image_id"] = old_to_new_id[ann["image_id"]]
        next_ann_id += 1
        merged_annotations.append(new_ann)

merged = {
    "images": merged_images,
    "annotations": merged_annotations,
    "categories": merged_categories,
}

out_path = BASE / "training_data" / "merged_instances.json"
with open(out_path, "w") as f:
    json.dump(merged, f)

print(f"Merge complete: {out_path}")
print(f"Total images: {len(merged_images)}")

# summarize split counts
from collections import Counter
split_counts = Counter((b["source_video"], b["split"]) for b in block_assignments)
for (src_tag, split), count in sorted(split_counts.items()):
    print(f"  {src_tag} / {split}: {count} images")

train_total = sum(1 for b in block_assignments if b["split"] == "train")
test_total = sum(1 for b in block_assignments if b["split"] == "test")
print(f"\nTotal train: {train_total}, test: {test_total}")

# also save split info as CSV for later reference
import csv
with open(BASE / "training_data" / "split_assignments.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image_id", "file_name", "source_video", "block_id", "split"])
    writer.writeheader()
    writer.writerows(block_assignments)
print("\nSaved split_assignments.csv as well")

Merge complete: ../training_data/merged_instances.json
Total images: 894
  video_A / test: 60 images
  video_A / train: 234 images
  video_B_seg1 / test: 60 images
  video_B_seg1 / train: 240 images
  video_B_seg2 / test: 60 images
  video_B_seg2 / train: 240 images

Total train: 714, test: 180

Saved split_assignments.csv as well


In [6]:
%run ../run_qsvm.py --merged_json ../training_data/merged_instances.json --image_dir ../images

[main] Loading kidney stone dataset (this downloads ResNet-18 ImageNet weights on first run — requires network access)
[data] train: 100 samples -> present=50, absent=50
[data] test: 40 samples -> present=20, absent=20
  [feature extraction] 16/100
  [feature extraction] 32/100
  [feature extraction] 48/100
  [feature extraction] 64/100
  [feature extraction] 80/100
  [feature extraction] 96/100
  [feature extraction] 100/100
  [feature extraction] 16/40
  [feature extraction] 32/40
  [feature extraction] 40/40
[data] PCA explained variance ratio (n_qubits=4): 0.373
[main] train=100 test=40
[main] train class balance: [50 50]  test class balance: [20 20]
[QSVM] Building quantum kernel (n_qubits=4, reps=2, backend=aer)
[QSVM] Computing training kernel matrix (100x100)... this is O(N^2) quantum circuit evaluations, may take a while.
[QSVM] Computing test kernel matrix (40x100)...
[QSVM] train_kernel_time=31.4s  test_kernel_time=23.6s  svc_fit_time=0.01s
[QSVM] Accuracy=0.450  F1=0.450  A

/opt/anaconda3/envs/qml_endo/lib/python3.11/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
